In [32]:
!apt install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 1s (57.4 kB/s)0m
Selecting previously unselected package tree.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [31]:
!pip install peft transformers[torch] trl bitsandbytes datasets -U --q

In [3]:
import os
import random
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training

In [4]:
dataset = load_dataset("flytech/python-codes-25k",split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

In [5]:
dataset

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

In [34]:
shuffled_dataset = dataset.shuffle(seed=42)
train_dataset = shuffled_dataset.select(range(100))
eval_dataset = shuffled_dataset.select(range(20))
test_dataset = shuffled_dataset.select(range(20))

In [17]:
print("="*100)
print(shuffled_dataset)
print("="*100)
print(shuffled_dataset[0]['input'])
print("="*100)
print(shuffled_dataset[0]['instruction'])
print("="*100)
print(shuffled_dataset[0]['text'])
print("="*100)
print(shuffled_dataset[0]['output'])
print("="*100)


Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order Let's roll! The ball's in our court! ```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```
```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```


In [18]:
# HF hub ID
BASE_MODEL_NAME = "arnir0/Tiny-LLM"

# Push adapter artifacts post training
OUTPUT_DIR = "./qlora-peft-output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

In [19]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/602 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/26.0M [00:00<?, ?B/s]

In [13]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Llama-style
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [20]:
from trl import SFTConfig, SFTTrainer

# Define ALL configuration parameters within SFTConfig
config = SFTConfig(
    # --- Standard TrainingArguments parameters ---
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    
    # --- SFT-specific parameters (moved here from previous iterations) ---
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

# Initialize the SFTTrainer, passing the SFTConfig object to the 'args' parameter
trainer = SFTTrainer(
    model=model,
    args=config,  # Pass the combined SFTConfig object here
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    # No other config parameters needed here
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

In [21]:
trainer.train()

Step,Training Loss
10,6.264300
20,6.253300


TrainOutput(global_step=25, training_loss=6.2785272216796875, metrics={'train_runtime': 2.2604, 'train_samples_per_second': 44.239, 'train_steps_per_second': 11.06, 'total_flos': 711918305280.0, 'train_loss': 6.2785272216796875, 'entropy': 5.71179850101471, 'num_tokens': 17280.0, 'mean_token_accuracy': 0.14945476688444614, 'epoch': 1.0})

In [24]:
ADAPTER_DIR = "./qlora-peft-output/adapter"

In [25]:
# -----------------------------
# 6. Save adapter weights
# -----------------------------
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

Adapter saved to: ./qlora-peft-output/adapter


In [27]:
!ls qlora-peft-output


adapter  checkpoint-25	README.md


In [28]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_ID = "meta-llama/Llama-3.1-8B-Instruct"
BASE_ID = "arnir0/Tiny-LLM"
ADAPTER_DIR = "./qlora-peft-output/adapter"
MERGED_DIR = "./merged-model"

# -------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------------------------------
# Load base model in fp16/bf16
# (merged model will be full precision LLM)
# -------------------------------------------------------
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# -------------------------------------------------------
# Attach adapter
# -------------------------------------------------------
print("Loading adapter onto base model...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# -------------------------------------------------------
# Merge adapter → base model
# -------------------------------------------------------
print("Merging LoRA adapter into base model weights...")
merged_model = model.merge_and_unload()   # <-- key line

# -------------------------------------------------------
# Save merged full model (optional)
# -------------------------------------------------------
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}\n")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...
Loading adapter onto base model...
Merging LoRA adapter into base model weights...

Merged model saved to: ./merged-model



In [33]:
!tree

.
├── merged-model
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   └── tokenizer.model
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── tokenizer.model
│   ├── checkpoint-25
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── tokenizer.model
│   │   ├── trainer_state.json
│   │   └── training_args.bin
│   └── README.md
└── sample_data
    ├── anscombe.json
    ├── california_housing_test.csv
    ├── california_housing_train.csv
    ├── mnist_test.csv
    ├── mnis

In [35]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# -------------------------------------------------------
# Run merged-model inference
# -------------------------------------------------------
print("==================== MERGED MODEL OUTPUT ====================\n")

for p in test_dataset:
    wrapped = (
        "You are a personalized assistant that knows details about the user based "
        "on prior fine-tuning data.\n\n"
        f"Question: {p['instruction']}\nAnswer:"
    )

    print(f"PROMPT: {p}\n")
    output = generate(wrapped)
    print(output)
    print("-" * 120)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


==================== MERGED MODEL OUTPUT ====================

PROMPT: {'output': '```python\nimport random\n\n# generating a list of unique numbers from 0 to 9 in random order\nrandom_numbers = random.sample(range(0, 10), 10)\n\n# sort list of numbers \nrandom_numbers.sort()\n\n# print sorted list of random numbers\nprint(random_numbers)\n# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]\n```', 'instruction': 'Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order', 'input': '', 'text': "Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order Let's roll! The ball's in our court! ```python\nimport random\n\n# generating a list of unique numbers from 0 to 9 in random order\nrandom_numbers = random.sample(range(0, 10), 10)\n\n# sort list of numbers \nrandom_numbers.sort()\n\n# print sorted list of random numbers\nprint(random_numbers)\n# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]\n```"}



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Answer: 100%
Question: What is the number of the number of variables?
Answer: What is the number of variables?
Answer: What is the number of variables?
Answer: What is the number of variables?
Answer: What is the number of variables?
Answer: What is the number of variables?
Answer: What is the number of variables?
Answer: What is the number of variables?
Answer: What is the variables in the variables?
Answer: What is the variables in the variables?
Answer: What is the variables in the variables?
Answer: What is the variables in the variables?
Answer: What is the variables in the variables?
Answer: What is the variables in the variables?
Answer: What is the variables in the variables?
Answer: What is the variables in the variables?
Answer: What is the variables in the variables?
Answe

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Analyse the given Python code and identify the errors, if any def set_variable(name, value):
    variable = name
    variable_value = value
    return variable_name
Answer:
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
-value =
------------------------------------------------------------------------------------------------------------------------
PROMPT: {'output': "```python\n#importing libraries\nimport nltk\nfrom nltk.tokenize import sent_tokenize,word_tokenize\nfrom nltk.stem import Wo

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Create a natural language processing (NLP) program in Python that extracts the main topic from a given sentence
Answer: 1.0.1.
Question: What is the answer to the question?
Answer: What is the answer to the question?
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer to the question
Answer: What is the answer 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Write a script in Python for generating a histogram for a given set of data. [3, 5, 8, 1, 9, 12]
Answer: 1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
------------------------------------------------------------------------------------------------------------------------
PROMPT: {'output': '"""\nWrite a Python program to create a KMeans model and cluster iris data into 3 clusters.\n"""\n\nimport pandas as pd\nfrom sklearn.cluster import KMeans\nfrom sklearn import datasets\n\n# Load the iris data\niris = datasets.load_iris()\nX = pd.DataFrame(iris.data)\ny = pd.DataFrame(iris.target)\n\n# Create the KMeans model \nmodel = KMeans(n_clusters=3, random_state=0)\nmodel.fit(X)\n\n# Predict the clusters\nprediction = model.predict(